**Feature Selection Method:** FLAML   
**Dataset:** QUT-DV25 (Combined)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [ ]:
# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()


In [ ]:
data.head()

,Package_Name,IO_Operations,File_Operations,Network_Operations,Time_Operations,Security_Operations,Process_Operations,Read_Processes,Write_Processes,Read_Data_Transfer,...,Pattern_2,Pattern_3,Pattern_4,Pattern_5,Pattern_6,Pattern_7,Pattern_8,Pattern_9,Pattern_10,Level
0,10Cent10-999.0.4.tar.gz,ioctl,NaN,"getrandom, uname",NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,fstat -> ioctl -> lseek,openat -> fstat -> ioctl,newfstatat -> newfstatat -> openat,newfstatat -> openat -> fstat,newfstatat -> newfstatat -> newfstatat -> no-e...,fstat -> ioctl -> lseek -> no-error -> no-fd,openat -> fstat -> ioctl -> no-error -> no-fd,ioctl -> lseek -> lseek -> error=ENOTTY -> no-fd,read -> read -> close -> no-error -> no-fd,1
1,10Cent11-999.0.4.tar.gz,"ioctl, poll",NaN,uname,NaN,NaN,"wait4, getpid","opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,ioctl -> ioctl -> ioctl,newfstatat -> newfstatat -> openat,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,ioctl -> ioctl -> ioctl -> no-error -> no-fd,fcntl -> fstat -> fcntl -> no-error -> no-fd,1
2,11Cent-999.0.0.tar.gz,"ioctl, poll",pipe2,"getrandom, uname",NaN,NaN,"vfork, wait4, getpid","opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,newfstatat -> newfstatat -> openat,read -> read -> read,newfstatat -> openat -> fstat,fstat -> ioctl -> lseek,newfstatat -> newfstatat -> newfstatat -> no-e...,newfstatat -> newfstatat -> openat -> no-error...,read -> read -> read -> no-error -> no-fd,openat -> fstat -> ioctl -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,1
3,11Cent-999.0.1.tar.gz,"ioctl, poll",NaN,uname,NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,openat -> fstat -> fcntl,fstat -> fcntl -> fstat,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,openat -> fstat -> fcntl -> no-error -> no-fd,fstat -> fcntl -> fstat -> no-error -> no-fd,1
4,11Cent-999.0.2.tar.gz,"poll, ioctl",NaN,uname,NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,read -> poll -> read,poll -> read -> read,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,read -> poll -> read -> error=EAGAIN -> no-fd,newfstatat -> newfstatat -> openat -> no-error...,1


In [ ]:
if 'Package_Name' in data.columns:
    data = data.drop(columns=['Package_Name'])

# Combine categorical columns into a single text feature
categorical_columns = [col for col in data.columns if data[col].dtype == 'object' and col != 'Level']
data['Combined_Categorical'] = data[categorical_columns].fillna('').astype(str).agg(' '.join, axis=1)

# Vectorize the combined categorical text column using n-grams
vectorizer = CountVectorizer(ngram_range=(2, 3))
categorical_ngrams = vectorizer.fit_transform(data['Combined_Categorical'])

# Ensure only valid numeric columns are selected
numerical_columns = [
    col for col in data.columns
    if pd.to_numeric(data[col], errors='coerce').notnull().all() and col != 'Level'
]

# Prepare numerical features
numerical_features = data[numerical_columns].fillna(0).astype(float)

# Convert numerical features to sparse matrix
numerical_features_sparse = csr_matrix(numerical_features.values)

# Combine numerical features with n-gram features
X = hstack([numerical_features_sparse, categorical_ngrams])

# Scale features before combining
scaler = StandardScaler(with_mean=False)  # Sparse matrices need `with_mean=False`
X_scaled = scaler.fit_transform(X)

<14271x168851 sparse matrix of type '<class 'numpy.float64'>'
	with 2862103 stored elements in Compressed Sparse Row format>

In [ ]:

X = data.drop(columns=['Level'])
y = data['Level']

if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize FLAML
automl = AutoML()
automl_settings = {
    "time_budget": 300,
    "metric": "accuracy",
    "task": "classification",
}

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# Feature importance & selection
selected_features = X.columns.tolist()
dropped_features = []

try:
    importances = automl.model.feature_importances_

    # Sometimes LGBM may drop unused features -> align with all columns
    if len(importances) < X.shape[1]:
        full_importances = np.zeros(X.shape[1])
        # Use feature names from model if available
        try:
            model_features = automl.model.booster_.feature_name()
            for i, col in enumerate(X.columns):
                if col in model_features:
                    idx = model_features.index(col)
                    full_importances[i] = importances[idx]
        except:
            full_importances[:len(importances)] = importances
        importances = full_importances

    varimp = pd.DataFrame({'variable': X.columns, 'importance': importances})
    varimp['relative_importance'] = varimp['importance'] / varimp['importance'].sum()

    threshold = 0.03
    selected_features = list(varimp[varimp['relative_importance'] > threshold]['variable'])
    dropped_features = list(varimp[varimp['relative_importance'] <= threshold]['variable'])

    print("Feature importance table:\n", varimp)

except AttributeError:
    print("Feature importance not available for the chosen model.")

print("\nSelected features:", selected_features)
print("Number of features kept:", len(selected_features))
print("\nDropped features:", dropped_features)
print("Number of features dropped:", len(dropped_features))


[flaml.automl.logger: 10-23 14:13:34] {1680} INFO - task = classification
[flaml.automl.logger: 10-23 14:13:34] {1691} INFO - Evaluation method: cv
[flaml.automl.logger: 10-23 14:13:34] {1789} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 10-23 14:13:35] {1901} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'lrl1']
[flaml.automl.logger: 10-23 14:13:35] {2219} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 10-23 14:13:36] {2346} INFO - Estimated sufficient time budget=15564s. Estimated necessary time budget=359s.
[flaml.automl.logger: 10-23 14:13:36] {2398} INFO -  at 1.9s,	estimator lgbm's best error=0.0271,	best estimator lgbm's best error=0.0271
[flaml.automl.logger: 10-23 14:13:36] {2219} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 10-23 14:13:38] {2398} INFO -  at 3.4s,	estimator lgbm's best error=0.0261,	best estimator lgbm's best error=0.0261
[flaml.automl.logger: 10-23 14:1

[flaml.automl.logger: 10-23 14:14:11] {2398} INFO -  at 37.0s,	estimator rf's best error=0.0358,	best estimator xgboost's best error=0.0051
[flaml.automl.logger: 10-23 14:14:11] {2219} INFO - iteration 34, current learner rf
[flaml.automl.logger: 10-23 14:14:13] {2398} INFO -  at 38.5s,	estimator rf's best error=0.0250,	best estimator xgboost's best error=0.0051
[flaml.automl.logger: 10-23 14:14:13] {2219} INFO - iteration 35, current learner rf
[flaml.automl.logger: 10-23 14:14:14] {2398} INFO -  at 40.2s,	estimator rf's best error=0.0250,	best estimator xgboost's best error=0.0051
[flaml.automl.logger: 10-23 14:14:14] {2219} INFO - iteration 36, current learner extra_tree
[flaml.automl.logger: 10-23 14:14:16] {2398} INFO -  at 42.3s,	estimator extra_tree's best error=0.0345,	best estimator xgboost's best error=0.0051
[flaml.automl.logger: 10-23 14:14:16] {2219} INFO - iteration 37, current learner xgboost
[flaml.automl.logger: 10-23 14:14:19] {2398} INFO -  at 45.1s,	estimator xgboos

[flaml.automl.logger: 10-23 14:16:30] {2219} INFO - iteration 69, current learner lgbm
[flaml.automl.logger: 10-23 14:16:38] {2398} INFO -  at 184.0s,	estimator lgbm's best error=0.0048,	best estimator lgbm's best error=0.0048
[flaml.automl.logger: 10-23 14:16:38] {2219} INFO - iteration 70, current learner rf
[flaml.automl.logger: 10-23 14:16:40] {2398} INFO -  at 185.6s,	estimator rf's best error=0.0233,	best estimator lgbm's best error=0.0048
[flaml.automl.logger: 10-23 14:16:40] {2219} INFO - iteration 71, current learner lgbm
[flaml.automl.logger: 10-23 14:16:44] {2398} INFO -  at 189.8s,	estimator lgbm's best error=0.0048,	best estimator lgbm's best error=0.0048
[flaml.automl.logger: 10-23 14:16:44] {2219} INFO - iteration 72, current learner extra_tree
[flaml.automl.logger: 10-23 14:16:46] {2398} INFO -  at 191.4s,	estimator extra_tree's best error=0.0258,	best estimator lgbm's best error=0.0048
[flaml.automl.logger: 10-23 14:16:46] {2219} INFO - iteration 73, current learner ex

[flaml.automl.logger: 10-23 14:18:14] {2398} INFO -  at 279.8s,	estimator rf's best error=0.0152,	best estimator lgbm's best error=0.0048
[flaml.automl.logger: 10-23 14:18:14] {2219} INFO - iteration 105, current learner rf
[flaml.automl.logger: 10-23 14:18:16] {2398} INFO -  at 281.4s,	estimator rf's best error=0.0129,	best estimator lgbm's best error=0.0048
[flaml.automl.logger: 10-23 14:18:16] {2219} INFO - iteration 106, current learner extra_tree
[flaml.automl.logger: 10-23 14:18:17] {2398} INFO -  at 283.3s,	estimator extra_tree's best error=0.0074,	best estimator lgbm's best error=0.0048
[flaml.automl.logger: 10-23 14:18:17] {2219} INFO - iteration 107, current learner rf
[flaml.automl.logger: 10-23 14:18:19] {2398} INFO -  at 284.9s,	estimator rf's best error=0.0120,	best estimator lgbm's best error=0.0048
[flaml.automl.logger: 10-23 14:18:19] {2219} INFO - iteration 108, current learner extra_tree
[flaml.automl.logger: 10-23 14:18:21] {2398} INFO -  at 286.5s,	estimator extra_

In [ ]:
selected_features = ['IO_Operations', 'File_Operations', 'Time_Operations', 'Process_Operations', 'Write_Processes', 'Read_Data_Transfer', 'Local_Port_Access', 'Remote_Port_Access', 'Total_Dependencies', 'Direct_Dependencies', 'Indirect_Dependencies', 'Pattern_1', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_6']